In [1]:
import os

dataset_path = "/kaggle/input/datasets/dhanushreerajangam/acrima-images-zip/Database/Images"

images = os.listdir(dataset_path)

print("Total images:", len(images))
print("First 10 images:")
print(images[:10])

Total images: 705
First 10 images:
['Im332_g_ACRIMA.jpg', 'Im414_g_ACRIMA.jpg', 'Im268_ACRIMA.jpg', 'Im280_ACRIMA.jpg', 'Im213_ACRIMA.jpg', 'Im194_ACRIMA.jpg', 'Im098_ACRIMA.jpg', 'Im463_g_ACRIMA.jpg', 'Im673_g_ACRIMA.jpg', 'Im600_g_ACRIMA.jpg']


In [2]:
!pip install -q timm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms

from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
import os
import numpy as np

In [8]:
from PIL import Image
from torch.utils.data import Dataset

class ACRIMADataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform

        self.images = [
            os.path.join(root_dir, f)
            for f in os.listdir(root_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ]

        self.images.sort()

        # 0 = Normal, 1 = Glaucoma
        self.labels = [
            1 if "_g_" in os.path.basename(img).lower() else 0
            for img in self.images
        ]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = Image.open(self.images[idx]).convert("RGB")
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label


# Load ACRIMA
dataset = ACRIMADataset(
    acrima_path,
    transform=transform
)

print("Total images:", len(dataset))
print("Glaucoma:", sum(dataset.labels))
print("Normal:", len(dataset.labels) - sum(dataset.labels))

Total images: 705
Glaucoma: 396
Normal: 309


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cuda
GPU: Tesla T4


In [5]:
num_epochs = 20
patience = 3

best_val_loss = float("inf")
patience_counter = 0

best_model_path = "/kaggle/working/best_convnext_fusion_regularized.pth"

print("Maximum epochs:", num_epochs)
print("Early stopping patience:", patience)

Maximum epochs: 20
Early stopping patience: 3


In [ ]:
CONVNEXT-TINY

In [14]:
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Transform loaded successfully!")

Transform loaded successfully!


In [15]:
# ConvNeXt-Tiny without MLFF

convnext_model = timm.create_model(
    "convnext_tiny",
    pretrained=False,
    num_classes=2
)

convnext_model = convnext_model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    convnext_model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

print("ConvNeXt-Tiny without MLFF ready")

ConvNeXt-Tiny without MLFF ready


In [16]:
best_val_acc = 0.0
patience = 3
counter = 0

for epoch in range(20):

    # Training
    convnext_model.train()
    train_correct = 0
    train_total = 0

    for images, labels_batch in train_loader:
        images = images.to(device)
        labels_batch = labels_batch.to(device)

        optimizer.zero_grad()

        outputs = convnext_model(images)
        loss = criterion(outputs, labels_batch)

        loss.backward()
        optimizer.step()

        preds = torch.argmax(outputs, dim=1)

        train_correct += (preds == labels_batch).sum().item()
        train_total += labels_batch.size(0)

    train_acc = train_correct / train_total

    # Validation
    convnext_model.eval()
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels_batch in val_loader:
            images = images.to(device)
            labels_batch = labels_batch.to(device)

            outputs = convnext_model(images)
            preds = torch.argmax(outputs, dim=1)

            val_correct += (preds == labels_batch).sum().item()
            val_total += labels_batch.size(0)

    val_acc = val_correct / val_total

    print(
        f"Epoch {epoch+1:02d}/20 | "
        f"Train Acc: {train_acc*100:.2f}% | "
        f"Val Acc: {val_acc*100:.2f}%"
    )

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        counter = 0

        torch.save(
            convnext_model.state_dict(),
            "/kaggle/working/best_convnext_without_mlff.pth"
        )

    else:
        counter += 1

        if counter >= patience:
            print("Early stopping")
            break

print(f"\nBest Validation Accuracy: {best_val_acc*100:.2f}%")

Epoch 01/20 | Train Acc: 56.56% | Val Acc: 75.89%
Epoch 02/20 | Train Acc: 79.26% | Val Acc: 76.60%
Epoch 03/20 | Train Acc: 82.27% | Val Acc: 77.30%
Epoch 04/20 | Train Acc: 83.69% | Val Acc: 80.85%
Epoch 05/20 | Train Acc: 83.51% | Val Acc: 82.98%
Epoch 06/20 | Train Acc: 84.93% | Val Acc: 82.27%
Epoch 07/20 | Train Acc: 85.28% | Val Acc: 78.72%
Epoch 08/20 | Train Acc: 86.70% | Val Acc: 78.01%
Early stopping

Best Validation Accuracy: 82.98%


In [17]:
# Load best model
convnext_model.load_state_dict(
    torch.load(
        "/kaggle/working/best_convnext_without_mlff.pth",
        map_location=device
    )
)

convnext_model.eval()

all_labels = []
all_preds = []

with torch.no_grad():
    for images, labels_batch in val_loader:
        images = images.to(device)

        outputs = convnext_model(images)
        preds = torch.argmax(outputs, dim=1)

        all_labels.extend(labels_batch.numpy())
        all_preds.extend(preds.cpu().numpy())

all_labels = np.array(all_labels)
all_preds = np.array(all_preds)

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)

TN, FP, FN, TP = cm.ravel()

accuracy = (TP + TN) / (TP + TN + FP + FN)
sensitivity = TP / (TP + FN)
specificity = TN / (TN + FP)
precision = TP / (TP + FP)
recall = sensitivity
f1 = 2 * precision * recall / (precision + recall)

print("ConvNeXt-Tiny WITHOUT MLFF")

print("\nConfusion Matrix:")
print(cm)

print(f"\nAccuracy    : {accuracy * 100:.2f}%")
print(f"Sensitivity : {sensitivity * 100:.2f}%")
print(f"Specificity : {specificity * 100:.2f}%")
print(f"Precision   : {precision * 100:.2f}%")
print(f"Recall      : {recall * 100:.2f}%")
print(f"F1 Score    : {f1 * 100:.2f}%")

ConvNeXt-Tiny WITHOUT MLFF

Confusion Matrix:
[[43 19]
 [ 5 74]]

Accuracy    : 82.98%
Sensitivity : 93.67%
Specificity : 69.35%
Precision   : 79.57%
Recall      : 93.67%
F1 Score    : 86.05%


In [ ]:
MAXVIT-TINY 

In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
import timm

from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
import os
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

Device: cuda


In [19]:
# MaxViT-Tiny without MLFF

maxvit_model = timm.create_model(
    "maxvit_tiny_tf_224",
    pretrained=False,
    num_classes=2
)

maxvit_model = maxvit_model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    maxvit_model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

print("MaxViT-Tiny without MLFF ready")

MaxViT-Tiny without MLFF ready


In [20]:
best_val_acc = 0.0
patience = 3
counter = 0

for epoch in range(20):

    # Training
    maxvit_model.train()
    train_correct = 0
    train_total = 0

    for images, labels_batch in train_loader:
        images = images.to(device)
        labels_batch = labels_batch.to(device)

        optimizer.zero_grad()

        outputs = maxvit_model(images)
        loss = criterion(outputs, labels_batch)

        loss.backward()
        optimizer.step()

        preds = torch.argmax(outputs, dim=1)

        train_correct += (preds == labels_batch).sum().item()
        train_total += labels_batch.size(0)

    train_acc = train_correct / train_total

    # Validation
    maxvit_model.eval()
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels_batch in val_loader:
            images = images.to(device)
            labels_batch = labels_batch.to(device)

            outputs = maxvit_model(images)
            preds = torch.argmax(outputs, dim=1)

            val_correct += (preds == labels_batch).sum().item()
            val_total += labels_batch.size(0)

    val_acc = val_correct / val_total

    print(
        f"Epoch {epoch+1:02d}/20 | "
        f"Train Acc: {train_acc*100:.2f}% | "
        f"Val Acc: {val_acc*100:.2f}%"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        counter = 0

        torch.save(
            maxvit_model.state_dict(),
            "/kaggle/working/best_maxvit_without_mlff.pth"
        )

    else:
        counter += 1

        if counter >= patience:
            print("Early stopping")
            break

print(f"\nBest Validation Accuracy: {best_val_acc*100:.2f}%")

Epoch 01/20 | Train Acc: 82.80% | Val Acc: 68.09%
Epoch 02/20 | Train Acc: 91.13% | Val Acc: 85.11%
Epoch 03/20 | Train Acc: 93.44% | Val Acc: 87.94%
Epoch 04/20 | Train Acc: 94.15% | Val Acc: 92.91%
Epoch 05/20 | Train Acc: 95.21% | Val Acc: 88.65%
Epoch 06/20 | Train Acc: 93.62% | Val Acc: 93.62%
Epoch 07/20 | Train Acc: 95.39% | Val Acc: 91.49%
Epoch 08/20 | Train Acc: 95.92% | Val Acc: 95.04%
Epoch 09/20 | Train Acc: 97.87% | Val Acc: 96.45%
Epoch 10/20 | Train Acc: 97.34% | Val Acc: 97.16%
Epoch 11/20 | Train Acc: 97.70% | Val Acc: 95.04%
Epoch 12/20 | Train Acc: 95.04% | Val Acc: 97.16%
Epoch 13/20 | Train Acc: 96.45% | Val Acc: 97.87%
Epoch 14/20 | Train Acc: 97.34% | Val Acc: 97.16%
Epoch 15/20 | Train Acc: 98.94% | Val Acc: 95.74%
Epoch 16/20 | Train Acc: 98.94% | Val Acc: 97.16%
Early stopping

Best Validation Accuracy: 97.87%


In [21]:
# Load best MaxViT model
maxvit_model.load_state_dict(
    torch.load(
        "/kaggle/working/best_maxvit_without_mlff.pth",
        map_location=device
    )
)

maxvit_model.eval()

maxvit_labels = []
maxvit_preds = []

with torch.no_grad():
    for images, labels_batch in val_loader:
        images = images.to(device)

        outputs = maxvit_model(images)
        preds = torch.argmax(outputs, dim=1)

        maxvit_labels.extend(labels_batch.numpy())
        maxvit_preds.extend(preds.cpu().numpy())

maxvit_labels = np.array(maxvit_labels)
maxvit_preds = np.array(maxvit_preds)

cm = confusion_matrix(maxvit_labels, maxvit_preds)

TN, FP, FN, TP = cm.ravel()

accuracy = (TP + TN) / (TP + TN + FP + FN)
sensitivity = TP / (TP + FN)
specificity = TN / (TN + FP)
precision = TP / (TP + FP)
recall = sensitivity
f1 = 2 * precision * recall / (precision + recall)

print("MaxViT-Tiny WITHOUT MLFF")

print("\nConfusion Matrix:")
print(cm)

print(f"\nAccuracy    : {accuracy * 100:.2f}%")
print(f"Sensitivity : {sensitivity * 100:.2f}%")
print(f"Specificity : {specificity * 100:.2f}%")
print(f"Precision   : {precision * 100:.2f}%")
print(f"Recall      : {recall * 100:.2f}%")
print(f"F1 Score    : {f1 * 100:.2f}%")

MaxViT-Tiny WITHOUT MLFF

Confusion Matrix:
[[61  1]
 [ 2 77]]

Accuracy    : 97.87%
Sensitivity : 97.47%
Specificity : 98.39%
Precision   : 98.72%
Recall      : 97.47%
F1 Score    : 98.09%


In [ ]:
SWIN TRANSFORMER

In [22]:
!pip install -q timm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
import timm

from torch.utils.data import Dataset, DataLoader, Subset
from PIL import Image
import os
import numpy as np
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [23]:
# Swin-Tiny without MLFF

swin_model = timm.create_model(
    "swin_tiny_patch4_window7_224",
    pretrained=False,
    num_classes=2
)

swin_model = swin_model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    swin_model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

print("Swin-Tiny without MLFF ready")

Swin-Tiny without MLFF ready


In [24]:
best_val_acc = 0.0
patience = 3
counter = 0

for epoch in range(20):

    swin_model.train()
    train_correct = 0
    train_total = 0

    for images, labels_batch in train_loader:
        images = images.to(device)
        labels_batch = labels_batch.to(device)

        optimizer.zero_grad()

        outputs = swin_model(images)
        loss = criterion(outputs, labels_batch)

        loss.backward()
        optimizer.step()

        preds = torch.argmax(outputs, dim=1)

        train_correct += (preds == labels_batch).sum().item()
        train_total += labels_batch.size(0)

    train_acc = train_correct / train_total

    swin_model.eval()
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels_batch in val_loader:
            images = images.to(device)
            labels_batch = labels_batch.to(device)

            outputs = swin_model(images)
            preds = torch.argmax(outputs, dim=1)

            val_correct += (preds == labels_batch).sum().item()
            val_total += labels_batch.size(0)

    val_acc = val_correct / val_total

    print(
        f"Epoch {epoch+1:02d}/20 | "
        f"Train Acc: {train_acc*100:.2f}% | "
        f"Val Acc: {val_acc*100:.2f}%"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        counter = 0

        torch.save(
            swin_model.state_dict(),
            "/kaggle/working/best_swin_without_mlff.pth"
        )

    else:
        counter += 1

        if counter >= patience:
            print("Early stopping")
            break

print(f"\nBest Validation Accuracy: {best_val_acc*100:.2f}%")

Epoch 01/20 | Train Acc: 49.82% | Val Acc: 58.16%
Epoch 02/20 | Train Acc: 63.30% | Val Acc: 74.47%
Epoch 03/20 | Train Acc: 84.40% | Val Acc: 80.85%
Epoch 04/20 | Train Acc: 86.52% | Val Acc: 83.69%
Epoch 05/20 | Train Acc: 88.48% | Val Acc: 85.11%
Epoch 06/20 | Train Acc: 90.25% | Val Acc: 85.11%
Epoch 07/20 | Train Acc: 89.36% | Val Acc: 84.40%
Epoch 08/20 | Train Acc: 87.94% | Val Acc: 85.11%
Early stopping

Best Validation Accuracy: 85.11%


In [25]:
# Load best Swin model

swin_model.load_state_dict(
    torch.load(
        "/kaggle/working/best_swin_without_mlff.pth",
        map_location=device
    )
)

swin_model.eval()

swin_labels = []
swin_preds = []

with torch.no_grad():
    for images, labels_batch in val_loader:
        images = images.to(device)

        outputs = swin_model(images)
        preds = torch.argmax(outputs, dim=1)

        swin_labels.extend(labels_batch.numpy())
        swin_preds.extend(preds.cpu().numpy())

swin_labels = np.array(swin_labels)
swin_preds = np.array(swin_preds)

cm = confusion_matrix(swin_labels, swin_preds)

TN, FP, FN, TP = cm.ravel()

accuracy = (TP + TN) / (TP + TN + FP + FN)
sensitivity = TP / (TP + FN)
specificity = TN / (TN + FP)
precision = TP / (TP + FP)
recall = sensitivity
f1 = 2 * precision * recall / (precision + recall)

print("Swin-Tiny WITHOUT MLFF")

print("\nConfusion Matrix:")
print(cm)

print(f"\nAccuracy    : {accuracy * 100:.2f}%")
print(f"Sensitivity : {sensitivity * 100:.2f}%")
print(f"Specificity : {specificity * 100:.2f}%")
print(f"Precision   : {precision * 100:.2f}%")
print(f"Recall      : {recall * 100:.2f}%")
print(f"F1 Score    : {f1 * 100:.2f}%")

Swin-Tiny WITHOUT MLFF

Confusion Matrix:
[[43 19]
 [ 2 77]]

Accuracy    : 85.11%
Sensitivity : 97.47%
Specificity : 69.35%
Precision   : 80.21%
Recall      : 97.47%
F1 Score    : 88.00%


In [ ]:
RESNET50 + MLFF

In [26]:
# ResNet50 without MLFF

resnet_model = models.resnet50(weights=None, num_classes=2)

resnet_model = resnet_model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    resnet_model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

print("ResNet50 without MLFF ready")

ResNet50 without MLFF ready


In [27]:
best_val_acc = 0.0
patience = 3
counter = 0

for epoch in range(20):

    resnet_model.train()
    train_correct = 0
    train_total = 0

    for images, labels_batch in train_loader:
        images = images.to(device)
        labels_batch = labels_batch.to(device)

        optimizer.zero_grad()

        outputs = resnet_model(images)
        loss = criterion(outputs, labels_batch)

        loss.backward()
        optimizer.step()

        preds = torch.argmax(outputs, dim=1)

        train_correct += (preds == labels_batch).sum().item()
        train_total += labels_batch.size(0)

    train_acc = train_correct / train_total

    resnet_model.eval()
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels_batch in val_loader:
            images = images.to(device)
            labels_batch = labels_batch.to(device)

            outputs = resnet_model(images)
            preds = torch.argmax(outputs, dim=1)

            val_correct += (preds == labels_batch).sum().item()
            val_total += labels_batch.size(0)

    val_acc = val_correct / val_total

    print(
        f"Epoch {epoch+1:02d}/20 | "
        f"Train Acc: {train_acc*100:.2f}% | "
        f"Val Acc: {val_acc*100:.2f}%"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        counter = 0

        torch.save(
            resnet_model.state_dict(),
            "/kaggle/working/best_resnet_without_mlff.pth"
        )

    else:
        counter += 1

        if counter >= patience:
            print("Early stopping")
            break

print(f"\nBest Validation Accuracy: {best_val_acc*100:.2f}%")

Epoch 01/20 | Train Acc: 70.04% | Val Acc: 52.48%
Epoch 02/20 | Train Acc: 89.18% | Val Acc: 70.92%
Epoch 03/20 | Train Acc: 92.20% | Val Acc: 88.65%
Epoch 04/20 | Train Acc: 92.73% | Val Acc: 90.07%
Epoch 05/20 | Train Acc: 95.74% | Val Acc: 56.74%
Epoch 06/20 | Train Acc: 92.55% | Val Acc: 94.33%
Epoch 07/20 | Train Acc: 93.79% | Val Acc: 87.94%
Epoch 08/20 | Train Acc: 94.50% | Val Acc: 92.91%
Epoch 09/20 | Train Acc: 96.99% | Val Acc: 88.65%
Early stopping

Best Validation Accuracy: 94.33%


In [28]:
# Load best ResNet50 model

resnet_model.load_state_dict(
    torch.load(
        "/kaggle/working/best_resnet_without_mlff.pth",
        map_location=device
    )
)

resnet_model.eval()

resnet_labels = []
resnet_preds = []

with torch.no_grad():
    for images, labels_batch in val_loader:
        images = images.to(device)

        outputs = resnet_model(images)
        preds = torch.argmax(outputs, dim=1)

        resnet_labels.extend(labels_batch.numpy())
        resnet_preds.extend(preds.cpu().numpy())

resnet_labels = np.array(resnet_labels)
resnet_preds = np.array(resnet_preds)

cm = confusion_matrix(resnet_labels, resnet_preds)

TN, FP, FN, TP = cm.ravel()

accuracy = (TP + TN) / (TP + TN + FP + FN)
sensitivity = TP / (TP + FN)
specificity = TN / (TN + FP)
precision = TP / (TP + FP)
recall = sensitivity
f1 = 2 * precision * recall / (precision + recall)

print("ResNet50 WITHOUT MLFF")

print("\nConfusion Matrix:")
print(cm)

print(f"\nAccuracy    : {accuracy * 100:.2f}%")
print(f"Sensitivity : {sensitivity * 100:.2f}%")
print(f"Specificity : {specificity * 100:.2f}%")
print(f"Precision   : {precision * 100:.2f}%")
print(f"Recall      : {recall * 100:.2f}%")
print(f"F1 Score    : {f1 * 100:.2f}%")

ResNet50 WITHOUT MLFF

Confusion Matrix:
[[62  0]
 [ 8 71]]

Accuracy    : 94.33%
Sensitivity : 89.87%
Specificity : 100.00%
Precision   : 100.00%
Recall      : 89.87%
F1 Score    : 94.67%


In [29]:
import os

print("Saved model files:")

for file in os.listdir("/kaggle/working"):
    if file.endswith(".pth"):
        print(file)

Saved model files:
best_swin_without_mlff.pth
best_maxvit_without_mlff.pth
best_resnet_without_mlff.pth
best_convnext_without_mlff.pth


In [30]:
import pandas as pd

ablation_results = pd.DataFrame([
    {
        "Model": "ConvNeXt-Tiny",
        "MLFF": "Without",
        "Accuracy": 82.98,
        "Sensitivity": 93.67,
        "Specificity": 69.35,
        "Precision": 79.57,
        "F1": 86.05
    },
    {
        "Model": "MaxViT-Tiny",
        "MLFF": "Without",
        "Accuracy": 97.87,
        "Sensitivity": 97.47,
        "Specificity": 98.39,
        "Precision": 98.72,
        "F1": 98.09
    },
    {
        "Model": "Swin-Tiny",
        "MLFF": "Without",
        "Accuracy": 85.11,
        "Sensitivity": 97.47,
        "Specificity": 69.35,
        "Precision": 80.21,
        "F1": 88.00
    },
    {
        "Model": "ResNet50",
        "MLFF": "Without",
        "Accuracy": 94.33,
        "Sensitivity": 89.87,
        "Specificity": 100.00,
        "Precision": 100.00,
        "F1": 94.67
    }
])

ablation_results.to_csv(
    "/kaggle/working/ablation_without_mlff.csv",
    index=False
)

print("Ablation results saved successfully.")

Ablation results saved successfully.
